## Introduction
From Kaggle: Twitter has become an important communication channel in times of emergency.
The ubiquitousness of smartphones enables people to announce an emergency they’re observing in real-time. Because of this, more agencies are interested in programatically monitoring Twitter (i.e. disaster relief organizations and news agencies).

But, it’s not always clear whether a person’s words are actually announcing a disaster. The goal of this analysis is to build a machine learning model that predicts which Tweets are about real disasters and which one’s aren’t.

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, SpatialDropout1D, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
import re

/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:98: UserWarning: unable to load libtensorflow_io_plugins.so: unable to open file: libtensorflow_io_plugins.so, from paths: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so']
caused by: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so: undefined symbol: _ZN3tsl6StatusC1EN10tensorflow5error4CodeESt17basic_string_viewIcSt11char_traitsIcEENS_14SourceLocationE']
  warnings.warn(f"unable to load libtensorflow_io_plugins.so: {e}")
/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:104: UserWarning: file system plugins are not loaded: unable to open file: l

# Load the Disaster Tweets
Let's have a look at the train and test dataset.

They contain:
- id
- keyword: A keyword from that tweet (although this may be blank!)
- location: The location the tweet was sent from (may also be blank)
- text: The text of a tweet
- target: 1 if the tweet is a real disaster or 0 if not

In [2]:
train_df = pd.read_csv("/kaggle/input/nlp-getting-started/train.csv")
test_df = pd.read_csv("/kaggle/input/nlp-getting-started/test.csv")

print('Training Set Shape = {}'.format(train_df.shape))
print('Training Set Memory Usage = {:.2f} MB'.format(train_df.memory_usage().sum() / 1024**2))
print('Test Set Shape = {}'.format(test_df.shape))
print('Test Set Memory Usage = {:.2f} MB'.format(test_df.memory_usage().sum() / 1024**2))

Training Set Shape = (7613, 5)
Training Set Memory Usage = 0.29 MB
Test Set Shape = (3263, 4)
Test Set Memory Usage = 0.10 MB


In [3]:
train_df.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [4]:
test_df.head()

,id,keyword,location,text
0,0,NaN,NaN,Just happened a terrible car crash
1,2,NaN,NaN,"Heard about #earthquake is different cities, s..."
2,3,NaN,NaN,"there is a forest fire at spot pond, geese are..."
3,9,NaN,NaN,Apocalypse lighting. #Spokane #wildfires
4,11,NaN,NaN,Typhoon Soudelor kills 28 in China and Taiwan


# Explore the dataset

In [5]:
train_df["length"] = train_df["text"].apply(lambda x : len(x))
test_df["length"] = test_df["text"].apply(lambda x : len(x))

print("Train Length Stat")
print(train_df["length"].describe())
print()

print("Test Length Stat")
print(test_df["length"].describe())

Train Length Stat
count    7613.000000
mean      101.037436
std        33.781325
min         7.000000
25%        78.000000
50%       107.000000
75%       133.000000
max       157.000000
Name: length, dtype: float64

Test Length Stat
count    3263.000000
mean      102.108183
std        33.972158
min         5.000000
25%        78.000000
50%       109.000000
75%       134.000000
max       151.000000
Name: length, dtype: float64


If you want to know more information about the data, you can grab useful information [here](https://www.kaggle.com/code/gunesevitan/nlp-with-disaster-tweets-eda-cleaning-and-bert)

Note that all the tweets are in english.

# Preprocess the data

In [6]:
def preprocess_text(text):
    """Cleans and preprocesses the tweet text."""
    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    # Remove punctuation and special characters
    text = re.sub(r'[^\w\s]', '', text)
    # Convert to lowercase
    text = text.lower()
    return text

train_df['text_cleaned'] = train_df['text'].apply(preprocess_text)
test_df['text_cleaned'] = test_df['text'].apply(preprocess_text)

### Tokenization and Word Embedding

In [7]:
# Parameters
MAX_WORDS = 10000  # Maximum number of words to keep in the vocabulary
MAX_LEN = 150      # Maximum length of a sequence
EMBEDDING_DIM = 100 # Dimension of the word embeddings

# Tokenize the text
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<oov>")
tokenizer.fit_on_texts(train_df['text_cleaned'])

# Convert text to sequences of integers
X = tokenizer.texts_to_sequences(train_df['text_cleaned'])
X_test = tokenizer.texts_to_sequences(test_df['text_cleaned'])

# Pad sequences to ensure uniform length
X = pad_sequences(X, maxlen=MAX_LEN)
X_test = pad_sequences(X_test, maxlen=MAX_LEN)

y = train_df['target'].values

# Split training data for validation
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Build and Train the Sequential Neural Network

In [8]:
model = Sequential([
    # 1. Embedding Layer: Converts integer sequences to dense vectors.
    Embedding(input_dim=MAX_WORDS, output_dim=EMBEDDING_DIM, input_length=MAX_LEN),

    # 2. SpatialDropout1D: Regularization to prevent overfitting. It drops entire 1D feature maps.
    SpatialDropout1D(0.2),

    # 3. Bidirectional LSTM: Processes the sequence in both forward and backward directions,
    Bidirectional(LSTM(64, dropout=0.2, recurrent_dropout=0.2)),

    # 4. Dense Output Layer: A single neuron with a sigmoid activation function
    Dense(1, activation='sigmoid')
])

# Compile the model
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()

# Train the model
history = model.fit(X_train, y_train,
                    epochs=5,
                    validation_data=(X_val, y_val),
                    batch_size=32)

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 150, 100)          1000000   
                                                                 
 spatial_dropout1d (SpatialD  (None, 150, 100)         0         
 ropout1D)                                                       
                                                                 
 bidirectional (Bidirectiona  (None, 128)              84480     
 l)                                                              
                                                                 
 dense (Dense)               (None, 1)                 129       
                                                                 
Total params: 1,084,609
Trainable params: 1,084,609
Non-trainable params: 0
_________________________________________________________________
Epoch 1/5
191/191 [===========================

## Results and further analysis

The training logs are suggesting that the model is starting to overfit. The training loss is consistently going down, and accuracy is going up but the validation starts to increase after the second epoch, and the val_accuracy peaks and then declines.

Some techniques I will implement to improve the model are to introduce early stopping and dropouts. With early stopping, the model now stops automatically when it reaches its best performance on the validation data. With dropouts, it's harder for the model to memorize the training data, which helps combat overfitting.

Hyperparameter tuning: Some hyperparameter I can conduct is changing the following, but for the second round of running the model, I'm going to hold off on this round to avoid changing too many items at once.

EMBEDDING_DIM: Smaller dimensions (e.g., 50)

LSTM/GRU Units: The number of neurons in the recurrent layers.

Learning Rate: The step size the optimizer takes.

Batch Size: The number of samples processed before the model is updated.

In [9]:
# Bidirectional LSTM with dropout
model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=EMBEDDING_DIM, input_length=MAX_LEN),
    SpatialDropout1D(0.3), # Increased dropout
    Bidirectional(LSTM(64, dropout=0.3, recurrent_dropout=0.3, return_sequences=True)),
    Bidirectional(LSTM(32, dropout=0.3, recurrent_dropout=0.3)),
    Dense(1, activation='sigmoid')
])


# Compile the model
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_1 (Embedding)     (None, 150, 100)          1000000   
                                                                 
 spatial_dropout1d_1 (Spatia  (None, 150, 100)         0         
 lDropout1D)                                                     
                                                                 
 bidirectional_1 (Bidirectio  (None, 150, 128)         84480     
 nal)                                                            
                                                                 
 bidirectional_2 (Bidirectio  (None, 64)               41216     
 nal)                                                            
                                                                 
 dense_1 (Dense)             (None, 1)                 65        
                                                      

### Training with early stopping

In [11]:
# Training with Early Stopping
from tensorflow.keras.callbacks import EarlyStopping
# 'patience=2' = wait for 2 epochs with no improvement before stopping.
early_stopping = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

# Train the model
history = model.fit(X_train, y_train,
                    epochs=10, # earlyStopping will prevent overfitting
                    validation_data=(X_val, y_val),
                    batch_size=32,
                    callbacks=[early_stopping]) # callback 


Epoch 1/10
191/191 [==============================] - 395s 2s/step - loss: 0.5583 - accuracy: 0.7044 - val_loss: 0.4674 - val_accuracy: 0.7919
Epoch 2/10
191/191 [==============================] - 377s 2s/step - loss: 0.3516 - accuracy: 0.8539 - val_loss: 0.4576 - val_accuracy: 0.7991
Epoch 3/10
191/191 [==============================] - 371s 2s/step - loss: 0.2552 - accuracy: 0.9018 - val_loss: 0.5341 - val_accuracy: 0.7899
Epoch 4/10
191/191 [==============================] - 371s 2s/step - loss: 0.1900 - accuracy: 0.9338 - val_loss: 0.5796 - val_accuracy: 0.7722


## Prediction and Submission

In [12]:
# Make predictions on the test set
predictions = model.predict(X_test)
# Convert probabilities to binary predictions (0 or 1)
binary_predictions = (predictions > 0.5).astype(int).flatten()

# Create the submission file
submission_df = pd.DataFrame({'id': test_df['id'], 'target': binary_predictions})
submission_df.to_csv('submission.csv', index=False)

102/102 [==============================] - 14s 126ms/step


## Discussion

Based on the new output, the model and the training process have improved because of addressing the overfitting problem. The model's best performance on the unseen validation data was at Epoch 2, with a val_loss of 0.4576 and val_accuracy of 0.7991. In the following epochs (3 and 4), the val_loss started to increase (0.5341 and 0.5796), indicating that the model was beginning to overfit.

Because patience was set to 2, the training process correctly identified this trend and stopped automatically after Epoch 4.

The main improvments were earlyStopping, which acted as a safeguard, preventing the model from continuing to train once its performance on the validation set started to degrade. The higher dropout rates and the stacked LSTM architecture helped make the model more robust.

If I were to run this model a third iteration, I would do some hyperparameter tuning like I described above. They are:

EMBEDDING_DIM: Change to 50

LSTM/GRU Units: The number of neurons in the recurrent layers, increase this.

Learning Rate: The step size the optimizer takes, I would lower rate.

Batch Size: The number of samples processed before the model is updated. I would Decrease this.